# Model 1 — Phase 6: Training, validation and test predictions

## 1. Install packages

In [1]:
%pip install -q bert-score rouge-score scikit-learn huggingface_hub

Note: you may need to restart the kernel to use updated packages.


## 2. Imports and settings

In [2]:
import copy
import gc
import json
import math
import random
import re
import time
import zipfile

from dataclasses import dataclass, asdict
from functools import lru_cache
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient
from sklearn.feature_extraction.text import TfidfVectorizer
from torch.nn.utils.rnn import pad_sequence
from torch.utils.checkpoint import checkpoint as activation_checkpoint
from torch.utils.data import DataLoader

MODEL_DIR = Path('/kaggle/input/datasets/tejasvaidya137/model1-foles-forphase6')
OUTPUT_DIR = Path('/kaggle/working/model1_phase6_simple_all')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
CONTEXT_LENGTH = 4096
CHUNK_LENGTH = 768
CHUNK_OVERLAP = 64
CHUNK_TARGET_TOKENS = 256       # Training budget, including EOS.
INTERMEDIATE_NEW_TOKENS = 96    # Short chunk summaries for the second stage.
FINAL_TARGET_TOKENS = 2048      # Training budget includes EOS; generation may stop earlier.

RANK = 16
ALPHA = 32
LORA_DROPOUT = 0.05
CHUNK_EPOCHS = 2
FINAL_EPOCHS = 6
CHUNK_LR = 1e-4
FINAL_LR = 5e-5
LABEL_SMOOTHING = 0.05
PATIENCE = 2

BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
CHUNK_INFERENCE_BATCH_SIZE = 8
USE_DATA_PARALLEL = True
USE_ACTIVATION_CHECKPOINTING = True
VALIDATION_DOCUMENTS = 100

NUM_BEAMS = 4
LENGTH_PENALTY = 0.8
NO_REPEAT_NGRAM_SIZE = 3
MIN_NEW_TOKENS = 0

CHUNK_PROMPT = (
    'Summarize this part of a legal judgment. Preserve important facts, issues, '
    'arguments, reasoning and decisions. Do not add unsupported facts. Judgment: '
)
FINAL_PROMPT = (
    'Write a coherent legal judgment summary from these intermediate summaries. '
    'Preserve facts, issues, arguments, court reasoning and final orders. '
    'Avoid repetition and unsupported facts. Intermediate summaries: '
)
SUMMARY_MARKER = ' Summary: '

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'
amp_dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16
HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')

def amp_context():
    return torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp)

print('Device:', device, '| GPUs:', torch.cuda.device_count())
print('Effective batch size:', BATCH_SIZE * GRAD_ACCUM_STEPS)

Device: cuda:0 | GPUs: 2
Effective batch size: 16


## 3. Tokenizer

In [3]:
SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class BytelevelBPE:

    def __init__(self, vocab_size=5000):
        self.vocab_size = vocab_size
        self.specials = SPECIALS.copy()
        self.byte_vocab = [str(i) for i in range(256)]
        self.merges = []
        self.merge_ranks = {}
        self.vocab = []
        self.stoi = {}
        self.itos = {}
        self.symbol_to_bytes = {}

    def word_to_symbols(self, word):
        byte_seq = word.encode('utf-8')
        return [str(b) for b in byte_seq] + ['</w>']

    @staticmethod
    def merge_symbols(symbols, pair, merged_symbol):
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == pair[0] and (symbols[i + 1] == pair[1]):
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        return tuple(new_symbols)

    def apply_bpe(self, word):
        symbols = self.word_to_symbols(word)
        while len(symbols) > 1:
            best_pair = None
            best_rank = float('inf')
            best_merged_symbol = None
            for i in range(len(symbols) - 1):
                pair = (symbols[i], symbols[i + 1])
                merge_info = self.merge_ranks.get(pair)
                if merge_info is not None:
                    rank, merged_symbol = merge_info
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = pair
                        best_merged_symbol = merged_symbol
            if best_pair is None:
                break
            symbols = list(self.merge_symbols(tuple(symbols), best_pair, best_merged_symbol))
        return symbols

    def encode(self, text, add_bos=False, add_eos=False):
        ids = []
        if add_bos:
            ids.append(self.stoi['<s>'])
        for word in text.split():
            symbols = self.apply_bpe(word)
            for symbol in symbols:
                ids.append(self.stoi.get(symbol, self.stoi['<unk>']))
        if add_eos:
            ids.append(self.stoi['</s>'])
        return ids

    def decode(self, ids):
        words = []
        current_bytes = bytearray()
        for token_id in ids:
            symbol = self.itos.get(token_id)
            if symbol is None:
                continue
            if symbol in self.specials:
                continue
            ends_word = self._symbol_ends_word(symbol)
            byte_value = self.symbol_to_bytes.get(symbol, b'')
            current_bytes.extend(byte_value)
            if ends_word:
                try:
                    word = bytes(current_bytes).decode('utf-8')
                    words.append(word)
                except UnicodeDecodeError:
                    words.append('<unk>')
                current_bytes = bytearray()
        if current_bytes:
            try:
                words.append(bytes(current_bytes).decode('utf-8'))
            except UnicodeDecodeError:
                words.append('<unk>')
        return ' '.join(words)

    def _symbol_ends_word(self, symbol):
        return self.word_ends.get(symbol, False)

    def get_vocab_size(self):
        return len(self.vocab)

## 4. Decoder-only model

In [4]:
class CausalSelfAttention(nn.Module):

    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float=0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        if self.d_head % 2 != 0:
            raise ValueError('RoPE requires an even head dimension.')
        inv_freq = 1.0 / 10000.0 ** (torch.arange(0, self.d_head, 2).float() / self.d_head)
        angles = torch.outer(torch.arange(max_seq_len).float(), inv_freq)
        self.register_buffer('rope_cos', angles.cos(), persistent=False)
        self.register_buffer('rope_sin', angles.sin(), persistent=False)
        self.attention_scale = 1.0 / math.sqrt(self.d_head) if attention_scale is None else float(attention_scale)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def apply_rope(self, q, k, offset=0):
        T = q.size(2)
        cos = self.rope_cos[offset:offset + T][None, None].to(q.dtype)
        sin = self.rope_sin[offset:offset + T][None, None].to(q.dtype)

        def rotate(x):
            first, second = x.chunk(2, dim=-1)
            return torch.cat([first * cos - second * sin, first * sin + second * cos], dim=-1)
        return (rotate(q), rotate(k))

    def forward(self, x, past=None, key_mask=None, use_cache=False):
        batch, length, width = x.shape
        q = self.q_proj(x).view(batch, length, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(batch, length, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(batch, length, self.n_heads, self.d_head).transpose(1, 2)
        offset = 0 if past is None else past[0].size(2)
        if offset + length > self.rope_cos.size(0):
            raise ValueError('Sequence exceeds the configured RoPE context.')
        q, k = self.apply_rope(q, k, offset)
        if past is not None:
            k = torch.cat([past[0], k], dim=2)
            v = torch.cat([past[1], v], dim=2)
        allowed = None
        causal = past is None and key_mask is None
        if not causal:
            query_positions = offset + torch.arange(length, device=x.device)
            key_positions = torch.arange(k.size(2), device=x.device)
            allowed = key_positions[None, :] <= query_positions[:, None]
            allowed = allowed[None, None]
            if key_mask is not None:
                allowed = allowed & key_mask[:, None, None, :]
        output = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, is_causal=causal, dropout_p=self.attn_dropout.p if self.training else 0.0, scale=self.attention_scale)
        output = output.transpose(1, 2).contiguous().view(batch, length, width)
        output = self.resid_dropout(self.out_proj(output))
        return (output, (k, v) if use_cache else None)

@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 512
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = 'rope'
    attention_scale: float = None
    norm_first: bool = False

class FeedForward(nn.Module):

    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout, attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.norm_first = cfg.norm_first

    def forward(self, x, past=None, key_mask=None, use_cache=False):
        if self.norm_first:
            attended, cache = self.attn(self.ln1(x), past, key_mask, use_cache)
            x = x + attended
            x = x + self.ffn(self.ln2(x))
        else:
            attended, cache = self.attn(x, past, key_mask, use_cache)
            x = self.ln1(x + attended)
            x = self.ln2(x + self.ffn(x))
        return (x, cache)

class DecoderLM(nn.Module):

    def __init__(self, cfg: LMConfig):
        super().__init__()
        if cfg.pos_emb_type != 'rope':
            raise ValueError('This notebook requires a RoPE model/checkpoint.')
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, input_ids, labels=None, past_key_values=None, key_mask=None, use_cache=False):
        x = self.tok_emb(input_ids)
        caches = []
        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            if self.training and USE_ACTIVATION_CHECKPOINTING and (not use_cache):
                x, cache = activation_checkpoint(block, x, use_reentrant=False)
            else:
                x, cache = block(x, past, key_mask, use_cache)
            if use_cache:
                caches.append(cache)
        x = self.ln_f(x)
        if labels is not None:
            valid = labels.ne(-100)
            logits = self.head(x[valid])
            loss = F.cross_entropy(logits.float(), labels[valid], reduction='sum', label_smoothing=LABEL_SMOOTHING)
            return (loss.reshape(1), valid.sum().reshape(1))
        if use_cache:
            return (self.head(x[:, -1]), caches)
        return self.head(x)

## 5. Load model settings and tokenizer

In [5]:
saved = torch.load(MODEL_DIR / 'model.pt', map_location='cpu', weights_only=True)
cfg = LMConfig(**saved['model_config'])
original_context = cfg.max_seq_len
cfg.max_seq_len = CONTEXT_LENGTH
del saved
state = json.loads((MODEL_DIR / 'tokenizer.json').read_text(encoding='utf-8'))

tokenizer = BytelevelBPE(vocab_size=state["vocab_size"])
tokenizer.specials = state["specials"]
tokenizer.vocab = state["vocab"]
tokenizer.vocab_size = len(tokenizer.vocab)

tokenizer.stoi = {symbol: i for i, symbol in enumerate(tokenizer.vocab)}
tokenizer.itos = {i: symbol for symbol, i in tokenizer.stoi.items()}

tokenizer.merges = [tuple(pair) for pair in state["merges"]]

tokenizer.merge_ranks = {
    tuple(record["pair"]): (
        record["rank"],
        record["merged_symbol"],
    )
    for record in state["merge_ranks"]
}

tokenizer.symbol_to_bytes = {
    symbol: bytes(values)
    for symbol, values in state["symbol_to_bytes"].items()
}

# Preserve the tokenizer's word-boundary decoding behavior.
word_ends = {"</w>": True}

for record in sorted(state["merge_ranks"], key=lambda row: row["rank"]):
    word_ends[record['merged_symbol']] = word_ends.get(record['pair'][1], False)

tokenizer.word_ends = word_ends

tokenizer.apply_bpe = lru_cache(maxsize=100000)(tokenizer.apply_bpe)

del state

PAD_ID = tokenizer.stoi["<pad>"]
UNK_ID = tokenizer.stoi["<unk>"]
BOS_ID = tokenizer.stoi["<s>"]
EOS_ID = tokenizer.stoi["</s>"]

assert cfg.vocab_size == tokenizer.get_vocab_size()
assert cfg.pos_emb_type == "rope"


CHUNK_PROMPT_IDS = tokenizer.encode(CHUNK_PROMPT)
FINAL_PROMPT_IDS = tokenizer.encode(FINAL_PROMPT)
MARKER_IDS = tokenizer.encode(SUMMARY_MARKER)
SEPARATOR_IDS = tokenizer.encode(' [...] ')

# Reserve the full target budget for both training and inference.
CHUNK_SOURCE_BUDGET = cfg.max_seq_len - 1 - len(CHUNK_PROMPT_IDS) - len(MARKER_IDS) - CHUNK_TARGET_TOKENS
FINAL_SOURCE_BUDGET = cfg.max_seq_len - 1 - len(FINAL_PROMPT_IDS) - len(MARKER_IDS) - FINAL_TARGET_TOKENS
CHUNK_LENGTH = min(CHUNK_LENGTH, CHUNK_SOURCE_BUDGET)
assert 0 <= CHUNK_OVERLAP < CHUNK_LENGTH
assert FINAL_SOURCE_BUDGET > 0
assert 1 <= INTERMEDIATE_NEW_TOKENS <= CHUNK_TARGET_TOKENS
print('Saved context:', original_context, '| New context:', cfg.max_seq_len)
print('Chunk tokens:', CHUNK_LENGTH, '| Final source budget:', FINAL_SOURCE_BUDGET)
print('Final summary limit:', FINAL_TARGET_TOKENS)

Saved context: 1024 | New context: 4096
Chunk tokens: 768 | Final source budget: 2008
Final summary limit: 2048


## 6. Read and split the data

In [ ]:
train_path = Path(hf_hub_download(repo_id='Exploration-Lab/CS-781-Capstone', filename='Phase_6_Summ/Train/train.json', repo_type='dataset', token=HF_TOKEN))
raw_records = json.loads(train_path.read_text(encoding='utf-8'))

def normalize_text(text):
    normalized = ' '.join(text.split())
    return normalized
records = []
seen_pairs = set()
for row in raw_records:
    document = row.get('document')
    summary = row.get('summary')
    if not isinstance(document, str) or not isinstance(summary, str) or (not document.strip()) or (not summary.strip()):
        continue
    document = document.strip()
    summary = summary.strip()
    document_key = normalize_text(document)
    pair = (document_key, normalize_text(summary))
    if pair in seen_pairs:
        continue
    seen_pairs.add(pair)
    records.append({'id': str(row['id']), 'document': document, 'summary': summary, 'document_key': document_key})
del raw_records
parent = list(range(len(records)))

def find(index):
    while parent[index] != index:
        parent[index] = parent[parent[index]]
        index = parent[index]
    return index
owners = {}
for i, row in enumerate(records):
    for key in (('id', row['id']), ('document', row['document_key'])):
        if key in owners:
            parent[find(i)] = find(owners[key])
        else:
            owners[key] = i
groups = [find(i) for i in range(len(records))]
unique_groups = sorted(set(groups))
if len(unique_groups) < 2:
    raise ValueError('At least two independent document groups are required.')
random.Random(SEED).shuffle(unique_groups)
n_dev = min(len(unique_groups) - 1, max(1, round(0.1 * len(unique_groups))))
dev_groups = set(unique_groups[:n_dev])
train_indices = [i for i, group in enumerate(groups) if group not in dev_groups]
dev_indices = [i for i, group in enumerate(groups) if group in dev_groups]
print('Training examples:', len(train_indices))
print('Validation examples:', len(dev_indices))
train_records = [records[i] for i in train_indices]
dev_indices = random.Random(SEED + 1).sample(dev_indices, min(VALIDATION_DOCUMENTS, len(dev_indices)))
dev_records = [records[i] for i in dev_indices]
test_path = Path(hf_hub_download(repo_id='Exploration-Lab/CS-781-Capstone', filename='Phase_6_Summ/Test/test.json', repo_type='dataset', token=HF_TOKEN))
test_records = json.loads(test_path.read_text(encoding='utf-8'))
for row in test_records:
    if 'id' not in row or not isinstance(row.get('document'), str):
        raise ValueError('A test record has no ID or document text.')

if set(train_indices) & set(dev_indices):
    raise ValueError('Training and validation overlap.')
(OUTPUT_DIR / 'split.json').write_text(json.dumps({
    'train_indices': train_indices, 'validation_indices': dev_indices, 'seed': SEED,
}, indent=2))
print('Documents:', len(train_records), 'train,', len(dev_records), 'validation,', len(test_records), 'test')

## 7. Prepare chunks and training examples

In [7]:
def split_chunks(document):
    ids = tokenizer.encode(document)
    chunks = []
    step = CHUNK_LENGTH - CHUNK_OVERLAP
    for start in range(0, len(ids), step):
        chunk = ids[start:start + CHUNK_LENGTH]
        chunks.append(chunk)
        if start + CHUNK_LENGTH >= len(ids):
            break
    return chunks or [[]]


def assign_sentences(chunks, summary):
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', summary) if s.strip()]
    assignments = [[] for _ in chunks]
    if not sentences:
        return assignments
    texts = [tokenizer.decode(chunk) for chunk in chunks]
    vectorizer = TfidfVectorizer(
        stop_words='english', ngram_range=(1, 2),
        sublinear_tf=True, max_features=30000,
    )
    try:
        vectors = vectorizer.fit_transform(texts + sentences)
    except ValueError:
        return assignments
    similarity = (vectors[len(chunks):] @ vectors[:len(chunks)].T).toarray()
    for sentence, scores in zip(sentences, similarity):
        assignments[int(scores.argmax())].append(sentence)
    return assignments


def make_chunk_rows(document_records):
    rows = []
    for i, row in enumerate(document_records):
        chunks = split_chunks(row['document'])
        targets = assign_sentences(chunks, row['summary'])
        for chunk, sentences in zip(chunks, targets):
            if sentences:
                rows.append({'source_ids': chunk, 'summary': ' '.join(sentences)})
        if (i + 1) % 500 == 0:
            print('Prepared chunk targets for', i + 1, 'documents')
    if not rows:
        raise ValueError('No usable chunk targets were created.')
    return rows


def make_prefix(source_ids, prompt_ids, target_limit):
    budget = cfg.max_seq_len - 1 - len(prompt_ids) - len(MARKER_IDS) - target_limit
    if budget < 1:
        raise ValueError('No context remains for source text.')
    return [BOS_ID] + prompt_ids + list(source_ids[:budget]) + MARKER_IDS


def make_example(source_ids, summary, prompt_ids, target_limit):
    prefix = make_prefix(source_ids, prompt_ids, target_limit)
    target = tokenizer.encode(summary)[:target_limit - 1] + [EOS_ID]
    tokens = prefix + target
    inputs = tokens[:-1]
    labels = [-100] * (len(prefix) - 1) + target
    assert len(inputs) == len(labels) <= cfg.max_seq_len
    return {
        'input_ids': torch.tensor(inputs, dtype=torch.long),
        'labels': torch.tensor(labels, dtype=torch.long),
    }


def collate(rows):
    return {
        'input_ids': pad_sequence([r['input_ids'] for r in rows], batch_first=True, padding_value=PAD_ID),
        'labels': pad_sequence([r['labels'] for r in rows], batch_first=True, padding_value=-100),
    }


def make_loader(rows, prompt_ids, target_limit, shuffle=False):
    examples = [make_example(r['source_ids'], r['summary'], prompt_ids, target_limit) for r in rows]
    return DataLoader(examples, batch_size=BATCH_SIZE, shuffle=shuffle,
                      collate_fn=collate, pin_memory=use_amp, num_workers=0,
                      generator=torch.Generator().manual_seed(SEED))

## 8. LoRA and model saving

In [8]:
class LoRA(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.base.requires_grad_(False)
        self.scale = ALPHA / RANK
        self.dropout = nn.Dropout(LORA_DROPOUT)
        self.A = nn.Linear(base.in_features, RANK, bias=False)
        self.B = nn.Linear(RANK, base.out_features, bias=False)
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return self.base(x) + self.scale * self.B(self.A(self.dropout(x)))

def new_stage_model():
    base = DecoderLM(copy.deepcopy(cfg))
    saved = torch.load(MODEL_DIR / 'model.pt', map_location='cpu', weights_only=True)
    base.load_state_dict(saved['model_state_dict'], strict=True)
    del saved
    base.requires_grad_(False)
    for block in base.blocks:
        for name in ['q_proj', 'k_proj', 'v_proj', 'out_proj']:
            setattr(block.attn, name, LoRA(getattr(block.attn, name)))
    return base.to(device)

def unwrap(model):
    return model.module if isinstance(model, nn.DataParallel) else model

def load_adapters(model, path):
    saved = torch.load(path, map_location='cpu', weights_only=True)
    base = unwrap(model)
    expected = {name for name, p in base.named_parameters() if p.requires_grad}
    assert set(saved['adapters']) == expected
    assert saved['model_config'] == asdict(cfg)
    assert saved['lora'] == [RANK, ALPHA, LORA_DROPOUT]
    base.load_state_dict(saved['adapters'], strict=False)
    base.eval()
    return base

def atomic_save(value, path, tensor_file=False):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    if tensor_file:
        torch.save(value, temporary)
    else:
        temporary.write_text(json.dumps(value, ensure_ascii=False), encoding='utf-8')
    temporary.replace(path)

def adapter_weights(model):
    return {name: p.detach().cpu().clone()
            for name, p in unwrap(model).named_parameters() if p.requires_grad}

def checkpoint_info(adapters, epoch, dev_loss):
    return {'adapters': adapters, 'model_config': asdict(cfg),
            'lora': [RANK, ALPHA, LORA_DROPOUT], 'epoch': epoch, 'dev_loss': dev_loss}

## 9. Training function

In [9]:
@torch.inference_mode()
def evaluate_loss(model, loader):
    model.eval()
    total_loss = total_tokens = 0
    for batch in loader:
        inputs = batch['input_ids'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        with amp_context():
            losses, counts = model(inputs, labels=labels)
        total_loss += losses.sum().item()
        total_tokens += counts.sum().item()
    return total_loss / max(total_tokens, 1)

def train_stage(base, train_loader, dev_loader, epochs, lr, name):
    parameters = [p for p in base.parameters() if p.requires_grad]
    if not parameters:
        raise ValueError('Load a training model with new_stage_model(), not a frozen decoding model.')
    optimizer = torch.optim.AdamW(parameters, lr=lr, weight_decay=0.01)
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp and amp_dtype == torch.float16)
    best_loss = float('inf')
    bad_epochs = 0
    history = []
    best_path = OUTPUT_DIR / f'{name}_best.pt'

    model = base
    if USE_DATA_PARALLEL and torch.cuda.device_count() > 1 and BATCH_SIZE > 1:
        model = nn.DataParallel(base, device_ids=list(range(min(BATCH_SIZE, torch.cuda.device_count()))))

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss = total_tokens = accumulated_tokens = 0
        started = time.perf_counter()
        scale = BATCH_SIZE * max(CHUNK_TARGET_TOKENS, FINAL_TARGET_TOKENS)
        for step, batch in enumerate(train_loader, 1):
            inputs = batch['input_ids'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)
            with amp_context():
                losses, counts = model(inputs, labels=labels)
                loss_sum = losses.sum()
            if not torch.isfinite(loss_sum):
                raise RuntimeError('Non-finite training loss.')
            scaler.scale(loss_sum / scale).backward()
            tokens = int(counts.sum().item())
            total_loss += loss_sum.item()
            total_tokens += tokens
            accumulated_tokens += tokens
            if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                for p in parameters:
                    if p.grad is not None:
                        p.grad.mul_(scale / accumulated_tokens)
                torch.nn.utils.clip_grad_norm_(parameters, 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                accumulated_tokens = 0
            if step % 100 == 0 or step == len(train_loader):
                print(f'{name} | epoch {epoch} | {step}/{len(train_loader)} | '
                      f'loss {total_loss / total_tokens:.4f}', flush=True)

        dev_loss = evaluate_loss(model, dev_loader)
        history.append({'epoch': epoch, 'train_loss': total_loss / total_tokens,
                        'dev_loss': dev_loss, 'minutes': (time.perf_counter() - started) / 60})
        print(history[-1])
        current = adapter_weights(base)
        if dev_loss < best_loss:
            best_loss, bad_epochs = dev_loss, 0
            atomic_save(checkpoint_info(current, epoch, dev_loss), best_path, True)
        else:
            bad_epochs += 1

        atomic_save(history, OUTPUT_DIR / f'{name}_history.json')
        if bad_epochs >= PATIENCE:
            print('Early stopping.')
            break
    return load_adapters(base, best_path)

## 10. Summary generation functions

Chunk summaries use greedy decoding. Final summaries use the single chosen beam-search setting.

In [10]:
def filter_logits(logits, generated, no_repeat_ngram_size=3, min_new_tokens=0):
    logits = logits.float().clone()
    logits[[PAD_ID, BOS_ID, UNK_ID]] = -float('inf')
    if len(generated) < min_new_tokens:
        logits[EOS_ID] = -float('inf')
    n = no_repeat_ngram_size
    if n > 0 and len(generated) >= n - 1:
        prefix = tuple(generated[-(n - 1):]) if n > 1 else ()
        banned = [generated[i + n - 1] for i in range(len(generated) - n + 1)
                  if tuple(generated[i:i + n - 1]) == prefix]
        if banned:
            logits[banned] = -float('inf')
    if not torch.isfinite(logits).any():
        logits[EOS_ID] = 0.0
    return logits


@torch.inference_mode()
def generate_chunks(model, prefixes, max_new_tokens):
    model.eval()
    width = max(len(p) for p in prefixes)
    assert width + max_new_tokens <= cfg.max_seq_len
    # Left padding puts every prompt's final token at the same column.
    ids = torch.full((len(prefixes), width), PAD_ID, dtype=torch.long, device=device)
    mask = torch.zeros_like(ids, dtype=torch.bool)
    for i, prefix in enumerate(prefixes):
        ids[i, -len(prefix):] = torch.tensor(prefix, device=device)
        mask[i, -len(prefix):] = True
    generated = [[] for _ in prefixes]
    finished = [False] * len(prefixes)
    with amp_context():
        logits, cache = model(ids, key_mask=mask, use_cache=True)
        for step in range(max_new_tokens):
            next_ids = []
            for i in range(len(prefixes)):
                token = EOS_ID if finished[i] else int(filter_logits(logits[i], generated[i], NO_REPEAT_NGRAM_SIZE, MIN_NEW_TOKENS).argmax())
                next_ids.append(token)
                if token == EOS_ID:
                    finished[i] = True
                elif not finished[i]:
                    generated[i].append(token)
            if all(finished) or step + 1 == max_new_tokens:
                break
            next_ids = torch.tensor(next_ids, device=device)[:, None]
            mask = torch.cat([mask, torch.ones((len(prefixes), 1), device=device, dtype=torch.bool)], dim=1)
            logits, cache = model(next_ids, past_key_values=cache, key_mask=mask, use_cache=True)
    return generated


@torch.inference_mode()
def generate_final(model, prefix, max_new_tokens=2048, num_beams=4,
                   length_penalty=0.8, no_repeat_ngram_size=3, min_new_tokens=0):
    model.eval()
    assert len(prefix) + max_new_tokens <= cfg.max_seq_len
    beams = [[]]
    scores = torch.zeros(1, device=device)
    completed = []
    with amp_context():
        ids = torch.tensor([prefix], device=device)
        logits, cache = model(ids, use_cache=True)
        for step in range(max_new_tokens):
            log_probs = torch.stack([
                F.log_softmax(filter_logits(logits[i], sequence, no_repeat_ngram_size, min_new_tokens), dim=-1)
                for i, sequence in enumerate(beams)
            ])
            candidates = scores[:, None] + log_probs
            width = min(2 * num_beams, candidates.numel())
            values, indices = candidates.flatten().topk(width)
            next_beams, next_scores, parents, next_tokens = [], [], [], []
            for value, index in zip(values.tolist(), indices.tolist()):
                if not math.isfinite(value):
                    continue
                parent, token = divmod(index, cfg.vocab_size)
                sequence = beams[parent] + [token]
                if token == EOS_ID:
                    normalized = value / (len(sequence) ** length_penalty)
                    completed.append((normalized, sequence))
                elif len(next_beams) < num_beams:
                    next_beams.append(sequence)
                    next_scores.append(value)
                    parents.append(parent)
                    next_tokens.append(token)
            completed = sorted(completed, key=lambda item: item[0], reverse=True)[:num_beams]
            beams = next_beams
            scores = torch.tensor(next_scores, device=device)
            if not beams or step + 1 == max_new_tokens:
                break
            # With nonnegative length penalty and nonpositive log-probabilities,
            # this is an optimistic bound for every unfinished beam.
            if len(completed) >= num_beams:
                upper_bound = float(scores.max()) / (max_new_tokens ** length_penalty)
                if completed[-1][0] >= upper_bound:
                    beams = []
                    break
            parent_ids = torch.tensor(parents, device=device)
            cache = [(k.index_select(0, parent_ids), v.index_select(0, parent_ids)) for k, v in cache]
            ids = torch.tensor(next_tokens, device=device)[:, None]
            logits, cache = model(ids, past_key_values=cache, use_cache=True)
    for sequence, score in zip(beams, scores.tolist()):
        completed.append((score / (max(1, len(sequence)) ** length_penalty), sequence))
    best = max(completed, key=lambda item: item[0])[1]
    return tokenizer.decode([token for token in best if token != EOS_ID])

## 11. Train the chunk model

In [ ]:
chunk_train_rows = make_chunk_rows(train_records)
chunk_dev_rows = make_chunk_rows(dev_records)
chunk_train_loader = make_loader(chunk_train_rows, CHUNK_PROMPT_IDS, CHUNK_TARGET_TOKENS, True)
chunk_dev_loader = make_loader(chunk_dev_rows, CHUNK_PROMPT_IDS, CHUNK_TARGET_TOKENS)
chunk_model = new_stage_model()
chunk_model = train_stage(chunk_model, chunk_train_loader, chunk_dev_loader,
                          CHUNK_EPOCHS, CHUNK_LR, 'chunk')
del chunk_train_rows, chunk_dev_rows, chunk_train_loader, chunk_dev_loader

## 12. Generate intermediate summaries

In [ ]:
def fit_intermediates(summaries, budget):
    parts = [list(ids) for ids in summaries if ids]
    if not parts:
        return []
    gap = SEPARATOR_IDS
    # Extremely long documents may have more chunks than the budget can represent.
    maximum_parts = max(1, (budget + len(gap)) // (len(gap) + 1))
    if len(parts) > maximum_parts:
        indices = np.linspace(0, len(parts) - 1, maximum_parts, dtype=int)
        parts = [parts[i] for i in indices]
    remaining = budget - (len(parts) - 1) * len(gap)
    sizes = [0] * len(parts)
    while remaining > 0:
        changed = False
        for i, part in enumerate(parts):
            if sizes[i] < len(part) and remaining:
                sizes[i] += 1
                remaining -= 1
                changed = True
        if not changed:
            break
    result = []
    for i, (part, size) in enumerate(zip(parts, sizes)):
        if i:
            result.extend(gap)
        result.extend(part[:size])
    assert len(result) <= budget
    return result

def make_intermediates(model, document_records, name):
    rows = []
    for i, row in enumerate(document_records):
        chunks = split_chunks(row['document'])
        prefixes = [make_prefix(chunk, CHUNK_PROMPT_IDS, CHUNK_TARGET_TOKENS)
                    for chunk in chunks]
        summaries = []
        for start in range(0, len(prefixes), CHUNK_INFERENCE_BATCH_SIZE):
            batch = prefixes[start:start + CHUNK_INFERENCE_BATCH_SIZE]
            summaries.extend(generate_chunks(model, batch, INTERMEDIATE_NEW_TOKENS))
        rows.append({
            'id': row['id'],
            'source_ids': fit_intermediates(summaries, FINAL_SOURCE_BUDGET),
            'summary': row.get('summary', ''),
        })
        if (i + 1) % 25 == 0 or i + 1 == len(document_records):
            atomic_save(rows, OUTPUT_DIR / f'{name}_intermediate_rows.json')
            print(f'{name}: generated {i + 1}/{len(document_records)}', flush=True)
    return rows


final_train_rows = make_intermediates(chunk_model, train_records, 'train')
final_dev_rows = make_intermediates(chunk_model, dev_records, 'validation')
final_test_rows = make_intermediates(chunk_model, test_records, 'test')

del chunk_model
gc.collect()
if use_amp:
    torch.cuda.empty_cache()

## 13. Train the final model

In [ ]:
final_train_loader = make_loader(final_train_rows, FINAL_PROMPT_IDS, FINAL_TARGET_TOKENS, True)
final_dev_loader = make_loader(final_dev_rows, FINAL_PROMPT_IDS, FINAL_TARGET_TOKENS)

final_model = new_stage_model()
final_model = train_stage(
    final_model, final_train_loader, final_dev_loader,
    FINAL_EPOCHS, FINAL_LR, 'final',
)

del final_model, final_train_loader, final_dev_loader
gc.collect()
if use_amp:
    torch.cuda.empty_cache()

## 14. Load the best final model

In [ ]:
final_model = new_stage_model()
final_model = load_adapters(final_model, OUTPUT_DIR / 'final_best.pt')
final_model.requires_grad_(False)
final_model.eval()
print('Loaded:', OUTPUT_DIR / 'final_best.pt')


def predict_summaries(model, rows, name):
    predictions = []
    for i, row in enumerate(rows):
        prefix = make_prefix(row['source_ids'], FINAL_PROMPT_IDS, FINAL_TARGET_TOKENS)
        summary = generate_final(
            model, prefix,
            max_new_tokens=FINAL_TARGET_TOKENS,
            num_beams=NUM_BEAMS,
            length_penalty=LENGTH_PENALTY,
            no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
            min_new_tokens=MIN_NEW_TOKENS,
        )
        predictions.append({'id': row['id'], 'summary': summary})
        atomic_save(predictions, OUTPUT_DIR / f'{name}_predictions.json')
        print(f'{name}: saved {i + 1}/{len(rows)}', flush=True)
    return predictions

## 15. Validate with the chosen decoding settings

Evaluate once on 100 validation documents: 4 beams, length penalty 0.8, and maximum 2,048 new tokens.

In [ ]:
from bert_score import BERTScorer
from rouge_score import rouge_scorer

# validation_predictions = predict_summaries(final_model, final_dev_rows, 'validation')
def score_predictions(predictions, rows, scorer):
    references = [r['summary'] for r in rows]
    candidates = [p['summary'] for p in predictions]
    precision, recall, f1 = scorer.score(candidates, references, batch_size=4)
    rouge = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    scores = [rouge.score(ref, pred) for ref, pred in zip(references, candidates)]
    return {
        'bertscore_precision': precision.mean().item(),
        'bertscore_recall': recall.mean().item(), 'bertscore_f1': f1.mean().item(),
        'rouge1': float(np.mean([s['rouge1'].fmeasure for s in scores])),
        'rougeL': float(np.mean([s['rougeL'].fmeasure for s in scores])),
        'mean_generated_tokens': float(np.mean([len(tokenizer.encode(s)) for s in candidates])),
        'bertscore_hash': scorer.hash,
    }

bert_device = 'cuda:1' if torch.cuda.device_count() > 1 else str(device)
scorer = BERTScorer(
    model_type='roberta-large', num_layers=17, lang='en',
    idf=False, rescale_with_baseline=False, device=bert_device, batch_size=4,
)
validation_results = {
    'documents': len(final_dev_rows),
    'num_beams': NUM_BEAMS,
    'length_penalty': LENGTH_PENALTY,
    'max_new_tokens': FINAL_TARGET_TOKENS,
    **score_predictions(validation_predictions, final_dev_rows, scorer),
}
atomic_save(validation_results, OUTPUT_DIR / 'validation_results.json')
print(validation_results)

del scorer
gc.collect()
if use_amp:
    torch.cuda.empty_cache()

## 16. Generate test predictions

In [ ]:
test_predictions = predict_summaries(final_model, final_test_rows, 'test')

## 17. Create the submission

In [ ]:
assert len(test_predictions) == len(test_records) and len(test_predictions) > 0
prediction_path = OUTPUT_DIR / 'prediction.json'
atomic_save(test_predictions, prediction_path)

submission_path = OUTPUT_DIR / 'submission.zip'
with zipfile.ZipFile(submission_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write(prediction_path, arcname='prediction.json')

print('Predictions:', prediction_path)
print('Submission:', submission_path)

# **Knowledge Distillation**

In [ ]:
# Attached outputs from your completed Phase 6 training run.
SAVED_PHASE6_DIR = OUTPUT_DIR

def read_saved_rows(filename):
    path = SAVED_PHASE6_DIR / filename
    return json.loads(path.read_text(encoding='utf-8'))

final_train_rows = read_saved_rows('train_intermediates.json')
final_dev_rows = read_saved_rows('validation_intermediates.json')
final_test_rows = read_saved_rows('test_intermediates.json')

print('Training:', len(final_train_rows))
print('Validation:', len(final_dev_rows))
print('Test:', len(final_test_rows))

Training: 6301
Validation: 100
Test: 100


In [19]:
from dataclasses import replace
import shutil

KD_DIR = OUTPUT_DIR / 'model1_kd'
KD_DIR.mkdir(parents=True, exist_ok=True)

TEACHER_PATH = SAVED_PHASE6_DIR / 'final_best.pt'

KD_TEMPERATURE = 2.0
KD_WEIGHT = 0.5

KD_EPOCHS = 3
KD_LR = 2e-5
KD_BATCH_SIZE = 1
KD_ACCUM_STEPS = 16
KD_PATIENCE = 2

STUDENT_LAYERS = max(1, cfg.n_layers // 2)

In [20]:
# Release the model loaded by the earlier notebook cells.
if 'final_model' in globals():
    del final_model

gc.collect()
if use_amp:
    torch.cuda.empty_cache()

teacher = new_stage_model()
teacher = load_adapters(teacher, TEACHER_PATH)
teacher.eval()

# Merge LoRA: merged weight = base weight + scaled update.
with torch.no_grad():
    for block in teacher.blocks:
        for name in ['q_proj', 'k_proj', 'v_proj', 'out_proj']:
            adapter = getattr(block.attn, name)
            linear = adapter.base

            linear.weight.add_(
                adapter.scale * (adapter.B.weight @ adapter.A.weight)
            )

            setattr(block.attn, name, linear)

teacher.requires_grad_(False)
teacher.eval()

DecoderLM(
  (tok_emb): Embedding(20000, 560)
  (blocks): ModuleList(
    (0-11): 12 x Block(
      (ln1): LayerNorm((560,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (q_proj): Linear(in_features=560, out_features=560, bias=False)
        (k_proj): Linear(in_features=560, out_features=560, bias=False)
        (v_proj): Linear(in_features=560, out_features=560, bias=False)
        (out_proj): Linear(in_features=560, out_features=560, bias=False)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln2): LayerNorm((560,), eps=1e-05, elementwise_affine=True)
      (ffn): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=560, out_features=2240, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=2240, out_features=560, bias=True)
          (3): Dropout(p=0.1, inplace=False)
        )
      )
    )
  )
  (ln_f): LayerNorm((560,)

In [21]:
assert 1 <= STUDENT_LAYERS < cfg.n_layers

student_cfg = replace(cfg, n_layers=STUDENT_LAYERS)
student = DecoderLM(student_cfg).to(device)

# Choose teacher layers spread across the original network.
if STUDENT_LAYERS == 1:
    selected_layers = [cfg.n_layers - 1]
else:
    selected_layers = torch.linspace(
        0, cfg.n_layers - 1, STUDENT_LAYERS
    ).round().long().tolist()

# Copy embeddings and final normalization.
student.tok_emb.load_state_dict(teacher.tok_emb.state_dict())
student.ln_f.load_state_dict(teacher.ln_f.state_dict())

# Copy selected decoder blocks.
for new_index, old_index in enumerate(selected_layers):
    student.blocks[new_index].load_state_dict(
        teacher.blocks[old_index].state_dict()
    )

# DecoderLM already ties the output head to the embeddings.
student.requires_grad_(True)

print('Teacher layers:', cfg.n_layers)
print('Student layers:', student_cfg.n_layers)
print('Copied teacher layers:', selected_layers)

print(
    'Teacher parameters:',
    sum(p.numel() for p in teacher.parameters())
)
print(
    'Student parameters:',
    sum(p.numel() for p in student.parameters())
)

Teacher layers: 12
Student layers: 6
Copied teacher layers: [0, 2, 4, 7, 9, 11]
Teacher parameters: 56420000
Student parameters: 33810560


In [22]:
def make_kd_loader(rows, shuffle=False):
    examples = [
        make_example(
            row['source_ids'],
            row['summary'],
            FINAL_PROMPT_IDS,
            FINAL_TARGET_TOKENS,
        )
        for row in rows
    ]

    return DataLoader(
        examples,
        batch_size=KD_BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=collate,
        pin_memory=use_amp,
        num_workers=0,
        generator=torch.Generator().manual_seed(SEED),
    )


kd_train_loader = make_kd_loader(
    final_train_rows,
    shuffle=True,
)

kd_dev_loader = make_kd_loader(final_dev_rows)

In [23]:
def kd_loss(student_logits, teacher_logits, targets):
    # Learn from the reference summary.
    ce_loss = F.cross_entropy(
        student_logits,
        targets,
        reduction='sum',
        label_smoothing=LABEL_SMOOTHING,
    )

    # Learn from the teacher's vocabulary distribution.
    student_log_probs = F.log_softmax(
        student_logits / KD_TEMPERATURE,
        dim=-1,
    )

    teacher_log_probs = F.log_softmax(
        teacher_logits / KD_TEMPERATURE,
        dim=-1,
    )

    soft_loss = F.kl_div(
        student_log_probs,
        teacher_log_probs,
        reduction='sum',
        log_target=True,
    ) * KD_TEMPERATURE ** 2

    return (
        (1 - KD_WEIGHT) * ce_loss
        + KD_WEIGHT * soft_loss
    )

In [24]:
def train_kd(student, teacher):
    optimizer = torch.optim.AdamW(
        student.parameters(),
        lr=KD_LR,
        weight_decay=0.01,
    )

    scaler = torch.amp.GradScaler(
        'cuda',
        enabled=use_amp and amp_dtype == torch.float16,
    )

    best_loss = float('inf')
    bad_epochs = 0
    history = []

    # Temporary scaling for stable gradient accumulation.
    scale = KD_BATCH_SIZE * FINAL_TARGET_TOKENS

    teacher.eval()

    for epoch in range(1, KD_EPOCHS + 1):
        student.train()
        optimizer.zero_grad(set_to_none=True)

        total_loss = 0
        total_tokens = 0
        accumulated_tokens = 0

        for step, batch in enumerate(kd_train_loader, 1):
            inputs = batch['input_ids'].to(
                device, non_blocking=True
            )
            labels = batch['labels'].to(
                device, non_blocking=True
            )

            # Only summary tokens and EOS contribute to loss.
            mask = labels.ne(-100)
            token_count = int(mask.sum().item())

            with torch.no_grad(), amp_context():
                teacher_logits = teacher(inputs)[mask].float()

            with amp_context():
                student_logits = student(inputs)[mask].float()

                loss = kd_loss(
                    student_logits,
                    teacher_logits,
                    labels[mask],
                )

            if not torch.isfinite(loss):
                raise RuntimeError('Non-finite KD loss.')

            scaler.scale(loss / scale).backward()

            total_loss += loss.item()
            total_tokens += token_count
            accumulated_tokens += token_count

            del teacher_logits, student_logits, loss

            update_now = (
                step % KD_ACCUM_STEPS == 0
                or step == len(kd_train_loader)
            )

            if update_now:
                scaler.unscale_(optimizer)

                # Average over actual target tokens in this group.
                for parameter in student.parameters():
                    if parameter.grad is not None:
                        parameter.grad.mul_(
                            scale / accumulated_tokens
                        )

                torch.nn.utils.clip_grad_norm_(
                    student.parameters(), 1.0
                )

                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

                accumulated_tokens = 0

            if step % 100 == 0 or step == len(kd_train_loader):
                print(
                    f'KD | epoch {epoch} | '
                    f'{step}/{len(kd_train_loader)} | '
                    f'loss {total_loss / total_tokens:.4f}',
                    flush=True,
                )

        # Reference-summary loss, without the KD term.
        dev_loss = evaluate_loss(student, kd_dev_loader)

        record = {
            'epoch': epoch,
            'train_kd_loss': total_loss / total_tokens,
            'dev_loss': dev_loss,
        }

        history.append(record)
        print(record)

        atomic_save(history, KD_DIR / 'history.json')

        if dev_loss < best_loss:
            best_loss = dev_loss
            bad_epochs = 0

            atomic_save(
                {
                    'model_config': asdict(student.cfg),
                    'model_state_dict': {
                        name: tensor.detach().cpu().clone()
                        for name, tensor
                        in student.state_dict().items()
                    },
                    'epoch': epoch,
                    'dev_loss': dev_loss,
                    'temperature': KD_TEMPERATURE,
                    'kd_weight': KD_WEIGHT,
                },
                KD_DIR / 'student_best.pt',
                tensor_file=True,
            )

        else:
            bad_epochs += 1

            if bad_epochs >= KD_PATIENCE:
                print('Early stopping.')
                break


train_kd(student, teacher)

shutil.copy2(
    MODEL_DIR / 'tokenizer.json',
    KD_DIR / 'tokenizer.json',
)

KD | epoch 1 | 100/6301 | loss 2.9428
KD | epoch 1 | 200/6301 | loss 2.8438
KD | epoch 1 | 300/6301 | loss 2.7865
KD | epoch 1 | 400/6301 | loss 2.7445
KD | epoch 1 | 500/6301 | loss 2.7047
KD | epoch 1 | 600/6301 | loss 2.6732
KD | epoch 1 | 700/6301 | loss 2.6506
KD | epoch 1 | 800/6301 | loss 2.6327
KD | epoch 1 | 900/6301 | loss 2.6146
KD | epoch 1 | 1000/6301 | loss 2.5976
KD | epoch 1 | 1100/6301 | loss 2.5852
KD | epoch 1 | 1200/6301 | loss 2.5726
KD | epoch 1 | 1300/6301 | loss 2.5624
KD | epoch 1 | 1400/6301 | loss 2.5537
KD | epoch 1 | 1500/6301 | loss 2.5455
KD | epoch 1 | 1600/6301 | loss 2.5409
KD | epoch 1 | 1700/6301 | loss 2.5344
KD | epoch 1 | 1800/6301 | loss 2.5279
KD | epoch 1 | 1900/6301 | loss 2.5201
KD | epoch 1 | 2000/6301 | loss 2.5147
KD | epoch 1 | 2100/6301 | loss 2.5094
KD | epoch 1 | 2200/6301 | loss 2.5033
KD | epoch 1 | 2300/6301 | loss 2.4984
KD | epoch 1 | 2400/6301 | loss 2.4931
KD | epoch 1 | 2500/6301 | loss 2.4877
KD | epoch 1 | 2600/6301 | loss 2.

PosixPath('/kaggle/working/model1_phase6_simple_all/model1_kd/tokenizer.json')

In [25]:
del teacher, student

gc.collect()
if use_amp:
    torch.cuda.empty_cache()

saved = torch.load(
    KD_DIR / 'student_best.pt',
    map_location='cpu',
    weights_only=True,
)

student_cfg = LMConfig(**saved['model_config'])

student = DecoderLM(student_cfg).to(device)
student.load_state_dict(
    saved['model_state_dict'],
    strict=True,
)

student.requires_grad_(False)
student.eval()

print('Loaded student epoch:', saved['epoch'])

del saved

Loaded student epoch: 3


In [26]:
NUM_BEAMS = 4
LENGTH_PENALTY = 0.8
FINAL_TARGET_TOKENS = 2048

kd_validation_predictions = predict_summaries(
    student,
    final_dev_rows,
    'kd_validation',
)

bert_device = (
    'cuda:1'
    if torch.cuda.device_count() > 1
    else str(device)
)

scorer = BERTScorer(
    model_type='roberta-large',
    num_layers=17,
    lang='en',
    idf=False,
    rescale_with_baseline=False,
    device=bert_device,
    batch_size=4,
)

kd_scores = score_predictions(
    kd_validation_predictions,
    final_dev_rows,
    scorer,
)

atomic_save(
    kd_scores,
    KD_DIR / 'validation_results.json',
)

print(kd_scores)

del scorer
gc.collect()

if use_amp:
    torch.cuda.empty_cache()

kd_validation: saved 1/100
kd_validation: saved 2/100
kd_validation: saved 3/100
kd_validation: saved 4/100
kd_validation: saved 5/100
kd_validation: saved 6/100
kd_validation: saved 7/100
kd_validation: saved 8/100
kd_validation: saved 9/100
kd_validation: saved 10/100
kd_validation: saved 11/100
kd_validation: saved 12/100
kd_validation: saved 13/100
kd_validation: saved 14/100
kd_validation: saved 15/100
kd_validation: saved 16/100
kd_validation: saved 17/100
kd_validation: saved 18/100
kd_validation: saved 19/100
kd_validation: saved 20/100
kd_validation: saved 21/100
kd_validation: saved 22/100
kd_validation: saved 23/100
kd_validation: saved 24/100
kd_validation: saved 25/100
kd_validation: saved 26/100
kd_validation: saved 27/100
kd_validation: saved 28/100
kd_validation: saved 29/100
kd_validation: saved 30/100
kd_validation: saved 31/100
kd_validation: saved 32/100
kd_validation: saved 33/100
kd_validation: saved 34/100
kd_validation: saved 35/100
kd_validation: saved 36/100
k

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'bertscore_precision': 0.8454603552818298, 'bertscore_recall': 0.825852632522583, 'bertscore_f1': 0.8354161977767944, 'rouge1': 0.3796894151953877, 'rougeL': 0.20211427876472768, 'mean_generated_tokens': 522.42, 'bertscore_hash': 'roberta-large_L17_no-idf_version=0.3.12(hug_trans=5.20.2)'}


In [27]:
kd_test_predictions = predict_summaries(
    student,
    final_test_rows,
    'kd_test',
)

assert (
    len(kd_test_predictions) == len(final_test_rows)
    and len(kd_test_predictions) > 0
)

prediction_path = KD_DIR / 'prediction.json'
atomic_save(kd_test_predictions, prediction_path)

with zipfile.ZipFile(
    KD_DIR / 'submission.zip',
    'w',
    zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        prediction_path,
        arcname='prediction.json',
    )

print('KD submission:', KD_DIR / 'submission.zip')

kd_test: saved 1/100
kd_test: saved 2/100
kd_test: saved 3/100
kd_test: saved 4/100
kd_test: saved 5/100
kd_test: saved 6/100
kd_test: saved 7/100
kd_test: saved 8/100
kd_test: saved 9/100
kd_test: saved 10/100
kd_test: saved 11/100
kd_test: saved 12/100
kd_test: saved 13/100
kd_test: saved 14/100
kd_test: saved 15/100
kd_test: saved 16/100
kd_test: saved 17/100
kd_test: saved 18/100
kd_test: saved 19/100
kd_test: saved 20/100
kd_test: saved 21/100
kd_test: saved 22/100
kd_test: saved 23/100
kd_test: saved 24/100
kd_test: saved 25/100
kd_test: saved 26/100
kd_test: saved 27/100
kd_test: saved 28/100
kd_test: saved 29/100
kd_test: saved 30/100
kd_test: saved 31/100
kd_test: saved 32/100
kd_test: saved 33/100
kd_test: saved 34/100
kd_test: saved 35/100
kd_test: saved 36/100
kd_test: saved 37/100
kd_test: saved 38/100
kd_test: saved 39/100
kd_test: saved 40/100
kd_test: saved 41/100
kd_test: saved 42/100
kd_test: saved 43/100
kd_test: saved 44/100
kd_test: saved 45/100
kd_test: saved 46/1